# 03 - Insert Chunks + Triples into Neo4j (ContraDoc, GraphRAG schema)

Ingests `data/processed/ContraDoc/triples.jsonl` under a **chunk-first GraphRAG schema**:

```
(:Document {doc_id, contradiction, doc_type, scope, contra_plug, contra_type,
            evidence, ref_sentences, gold_evidence_sentence_id, gold_ref_sentence_ids})
(:Chunk {doc_id, sentence_id, source_text, embedding,
         is_gold_evidence, is_gold_ref, ref_index})
(:Entity {doc_id, name})

(:Document)-[:CONTAINS]->(:Chunk)
(:Chunk)-[:MENTIONS]->(:Entity)
(:Entity)-[:RELATION {predicate, sentence_id, doc_id,
                      is_gold_evidence, is_gold_ref, ref_index}]->(:Entity)
```

**Why this shape.** The `:Chunk` carries the natural-language sentence + its SBERT embedding. Vector search at step 05 operates on chunks (not triples), so one vector per sentence instead of one per claim. The `:Entity` / `:RELATION` subgraph is untouched — step 04's S-SR / S-SO queries keep working. `:MENTIONS` bridges unstructured text to structured facts (canonical GraphRAG pattern), enabling hybrid queries like 'chunks near X in embedding space that share an entity with Y'.

**Input:** `data/processed/ContraDoc/triples.jsonl` (output of notebook 02).

**Effect:** wipes any prior `:Document` / `:Chunk` / `:Entity` nodes (scoped clear), re-inserts fresh, creates a vector index on `:Chunk.embedding`.

In [ ]:
import json
from pathlib import Path

from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

from config import settings

INPUT_PATH = Path("data/processed/ContraDoc/triples.jsonl")
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 384

## Connect to Neo4j

In [2]:
driver = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password.get_secret_value()),
)
driver.verify_connectivity()


def run(cypher, **params):
    with driver.session() as s:
        return list(s.run(cypher, **params))


n_nodes = run("MATCH (n) RETURN count(n) AS n")[0]["n"]
n_rels = run("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"]
print(f"Connected. Existing nodes: {n_nodes}, relationships: {n_rels}")

Connected. Existing nodes: 14491, relationships: 22634


## Clear prior ContraDoc data (scoped)

Deletes only `:Document`, `:Chunk`, `:Entity` nodes (and their relationships). Other graph content is untouched.

In [3]:
CLEAR_CONTRADOC = True

if CLEAR_CONTRADOC:
    for label in ("Chunk", "Entity", "Document"):
        run(f"MATCH (n:{label}) DETACH DELETE n")
    print("Cleared :Document, :Chunk, :Entity nodes (and their relationships).")
else:
    print("CLEAR_CONTRADOC=False — skipping clear.")

Cleared :Document, :Chunk, :Entity nodes (and their relationships).


## Load `triples.jsonl` and flatten to chunks

Each record becomes one document with a list of chunks (one per sentence). Each chunk carries its triples and the derived gold flags.

In [4]:
records = [json.loads(line) for line in INPUT_PATH.open(encoding="utf-8")]
print(f"Loaded {len(records)} documents from {INPUT_PATH}")


def flatten(rec):
    ev_id = rec.get("gold_evidence_sentence_id")
    ref_ids = rec.get("gold_ref_sentence_ids") or []
    ref_index_by_sid = {sid: i for i, sid in enumerate(ref_ids)}
    chunks = []
    for sent in rec["sentences"]:
        sid = sent["sentence_id"]
        is_ev = sid == ev_id
        ref_idx = ref_index_by_sid.get(sid, -1)
        is_ref = ref_idx != -1
        chunks.append(
            {
                "sentence_id": sid,
                "source_text": sent["source_text"],
                "is_gold_evidence": is_ev,
                "is_gold_ref": is_ref,
                "ref_index": ref_idx,
                "triples": sent["triples"],
            }
        )
    return {
        "doc_id": rec["doc_id"],
        "doc_props": {
            "doc_id": rec["doc_id"],
            "contradiction": rec["contradiction"],
            "doc_type": rec["doc_type"],
            "scope": rec.get("scope") or "none",
            "contra_plug": rec.get("contra_plug") or "none",
            "contra_type": rec.get("contra_type") or "none",
            "evidence": rec.get("evidence") or "",
            "ref_sentences": rec.get("ref_sentences") or "none",
            "gold_evidence_sentence_id": ev_id if ev_id is not None else -1,
            "gold_ref_sentence_ids": ref_ids,
        },
        "chunks": chunks,
    }


rows = [flatten(r) for r in records]
n_chunks = sum(len(r["chunks"]) for r in rows)
n_triples = sum(sum(len(c["triples"]) for c in r["chunks"]) for r in rows)
print(f"Flattened: {len(rows)} docs, {n_chunks} chunks, {n_triples} triples")

Loaded 100 documents from data\processed\ContraDoc\triples.jsonl
Flattened: 100 docs, 3664 chunks, 7129 triples


## Embed every chunk with SBERT

One encode call over all sentences in the corpus. Embeddings are L2-normalized so cosine-via-dot-product works natively with Neo4j's vector index.

In [5]:
model = SentenceTransformer(SBERT_MODEL)

all_texts = []
chunk_refs = []
for row in rows:
    for ch in row["chunks"]:
        all_texts.append(ch["source_text"])
        chunk_refs.append(ch)

embeddings = model.encode(
    all_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
for ch, emb in zip(chunk_refs, embeddings):
    ch["embedding"] = emb.tolist()

print(f"Encoded {len(all_texts)} chunks -> {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/58 [00:00<?, ?it/s]

Encoded 3664 chunks -> (3664, 384)


## Constraints, indexes, and the chunk vector index

Idempotent. The vector index enables Cypher-native ANN search on `:Chunk.embedding` — used by step 05 and any hybrid GraphRAG query.

In [6]:
run("CREATE CONSTRAINT document_doc_id IF NOT EXISTS FOR (d:Document) REQUIRE d.doc_id IS UNIQUE")
run("CREATE CONSTRAINT chunk_doc_sid IF NOT EXISTS FOR (c:Chunk) REQUIRE (c.doc_id, c.sentence_id) IS UNIQUE")
run("CREATE CONSTRAINT entity_doc_name IF NOT EXISTS FOR (e:Entity) REQUIRE (e.doc_id, e.name) IS UNIQUE")
run("CREATE INDEX chunk_doc_id IF NOT EXISTS FOR (c:Chunk) ON (c.doc_id)")
run("CREATE INDEX relation_doc_id IF NOT EXISTS FOR ()-[r:RELATION]-() ON (r.doc_id)")
run("CREATE INDEX relation_predicate IF NOT EXISTS FOR ()-[r:RELATION]-() ON (r.predicate)")
run(
    f"CREATE VECTOR INDEX chunk_embedding IF NOT EXISTS "
    f"FOR (c:Chunk) ON (c.embedding) "
    f"OPTIONS {{ indexConfig: {{ `vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine' }} }}"
)
print("Constraints + indexes + vector index created (or already existed).")

Constraints + indexes + vector index created (or already existed).


## Insert documents, chunks, entities, and triple edges

Per-document `UNWIND`. Entities are MERGEd (scoped per doc); chunks carry their embedding as a property. `:MENTIONS` links chunk → subject and chunk → object of each triple. `:RELATION` edges carry predicate + provenance.

In [7]:
INSERT_QUERY = """
MERGE (d:Document {doc_id: $doc_id})
SET d += $doc_props
WITH d
UNWIND $chunks AS ch
  MERGE (c:Chunk {doc_id: $doc_id, sentence_id: ch.sentence_id})
  SET c.source_text = ch.source_text,
      c.embedding = ch.embedding,
      c.is_gold_evidence = ch.is_gold_evidence,
      c.is_gold_ref = ch.is_gold_ref,
      c.ref_index = ch.ref_index
  MERGE (d)-[:CONTAINS]->(c)
  WITH d, c, ch
  UNWIND ch.triples AS t
    MERGE (s:Entity {doc_id: $doc_id, name: t.s})
    MERGE (o:Entity {doc_id: $doc_id, name: t.o})
    MERGE (c)-[:MENTIONS]->(s)
    MERGE (c)-[:MENTIONS]->(o)
    CREATE (s)-[:RELATION {
      predicate: t.p,
      sentence_id: ch.sentence_id,
      doc_id: $doc_id,
      is_gold_evidence: ch.is_gold_evidence,
      is_gold_ref: ch.is_gold_ref,
      ref_index: ch.ref_index
    }]->(o)
"""

with driver.session() as session:
    for row in tqdm(rows):
        session.run(INSERT_QUERY, doc_id=row["doc_id"], doc_props=row["doc_props"], chunks=row["chunks"])

print("Insert complete.")

  0%|          | 0/100 [00:00<?, ?it/s]

Insert complete.


## Report-ready statistics

In [8]:
print("=== Node and relationship counts ===")
for label, cypher in [
    ("Documents", "MATCH (d:Document) RETURN count(d) AS n"),
    ("Chunks", "MATCH (c:Chunk) RETURN count(c) AS n"),
    ("Entities", "MATCH (e:Entity) RETURN count(e) AS n"),
    (":CONTAINS", "MATCH ()-[r:CONTAINS]->() RETURN count(r) AS n"),
    (":MENTIONS", "MATCH ()-[r:MENTIONS]->() RETURN count(r) AS n"),
    (":RELATION", "MATCH ()-[r:RELATION]->() RETURN count(r) AS n"),
]:
    n = run(cypher)[0]["n"]
    print(f"  {label:12s}: {n}")
print()

print("=== Per-doc averages ===")
r = run(
    "MATCH (d:Document)-[:CONTAINS]->(c:Chunk) "
    "WITH d, count(c) AS n_chunks "
    "OPTIONAL MATCH ()-[r:RELATION {doc_id: d.doc_id}]->() "
    "WITH d, n_chunks, count(r) AS n_rels "
    "OPTIONAL MATCH (d)-[:CONTAINS]->(:Chunk)-[:MENTIONS]->(e:Entity) "
    "WITH d, n_chunks, n_rels, count(DISTINCT e) AS n_entities "
    "RETURN avg(n_chunks) AS avg_chunks, avg(n_rels) AS avg_rels, avg(n_entities) AS avg_entities, "
    "       max(n_rels) AS max_rels, min(n_rels) AS min_rels"
)[0]
print(f"  Avg chunks/doc:   {r['avg_chunks']:.1f}")
print(f"  Avg entities/doc: {r['avg_entities']:.1f}")
print(f"  Avg triples/doc:  {r['avg_rels']:.1f}  (min={r['min_rels']}, max={r['max_rels']})")
print()

print("=== Gold coverage on chunks ===")
for label, cypher in [
    ("Chunks flagged is_gold_evidence", "MATCH (c:Chunk {is_gold_evidence: true}) RETURN count(c) AS n, count(DISTINCT c.doc_id) AS d"),
    ("Chunks flagged is_gold_ref", "MATCH (c:Chunk {is_gold_ref: true}) RETURN count(c) AS n, count(DISTINCT c.doc_id) AS d"),
]:
    row = run(cypher)[0]
    print(f"  {label}: {row['n']} chunks across {row['d']} docs")
print()

print("=== Sample gold chunk pair ===")
sample = run(
    "MATCH (d:Document {contradiction: 'YES'})-[:CONTAINS]->(ev:Chunk {is_gold_evidence: true}) "
    "MATCH (d)-[:CONTAINS]->(ref:Chunk {is_gold_ref: true}) "
    "RETURN d.doc_id AS doc_id, ev.sentence_id AS ev_sid, ev.source_text AS ev_text, "
    "       ref.sentence_id AS ref_sid, ref.source_text AS ref_text LIMIT 1"
)
for r in sample:
    print(f"  doc_id={r['doc_id']}")
    print(f"  EVIDENCE [sid={r['ev_sid']}]: {r['ev_text'][:140]}")
    print(f"  REF      [sid={r['ref_sid']}]: {r['ref_text'][:140]}")

=== Node and relationship counts ===
  Documents   : 100
  Chunks      : 3664
  Entities    : 9529
  :CONTAINS   : 3664
  :MENTIONS   : 11811
  :RELATION   : 7129

=== Per-doc averages ===
  Avg chunks/doc:   36.6
  Avg entities/doc: 95.3
  Avg triples/doc:  71.3  (min=20, max=146)

=== Gold coverage on chunks ===
  Chunks flagged is_gold_evidence: 38 chunks across 38 docs
  Chunks flagged is_gold_ref: 44 chunks across 43 docs

=== Sample gold chunk pair ===
  doc_id=3499318673_1
  EVIDENCE [sid=46]: Little Gidding is the first poem of T. S. Eliot 's Four Quartets.
  REF      [sid=2]: Little Gidding is the fourth and final poem of T. S. Eliot 's Four Quartets , a series of poems that discuss time , perspective , humanity ,


In [9]:
driver.close()
print("Driver closed.")

Driver closed.
